# COMP9085 Week 6 Laboratory
## Preparing NHANES without breaking it

**Two hours.** Last week the problems that mattered most were data problems. This
week you take a real public-health survey from nine raw files to a modelling-ready
matrix and make every preparation decision explicitly: what one row is, what the
codes mean, why values are missing, which unusual values to keep, how to encode
and scale, how to handle an imbalanced outcome, and whom the sample describes.

**Data.** The US National Health and Nutrition Examination Survey (NHANES),
August 2021 – August 2023 cycle, public-use files from the CDC National Center for
Health Statistics. The nine CSV files in `data/` next to this notebook are column
extracts of the released files, one row per participant, linked by the
participant number `SEQN`. Values are exactly as released, including the codes
for "refused" and "don't know"; `data/README.md` documents every column and
`data/PROVENANCE.json` the source and checksums. **No network is needed.**

**The question.** Among adults aged 20 or over who had blood drawn, can routinely
available information (demographics, body measures, blood pressure, cholesterol,
one day of diet, smoking history) identify those whose **HbA1c is 6.5% or
higher**, the diabetes-range threshold? A positive prediction would at most
suggest a confirmatory test. **A prediction is not a diagnosis**, and nothing in
this lab is a clinical tool.

**Who is in the data.** NHANES samples the US civilian non-institutionalised
population. In this lab an adult is **eligible** if aged 20 or over (as recorded;
80 means 80 or older). The **modelling population** is the eligible adults with an
HbA1c result, which requires attending the mobile examination centre and giving a
blood sample. Task 1 counts who is lost at each step; Task 3 asks who is missing
the outcome and whether that matters.

**By the end of this lab you should be able to**
1. merge survey files on a participant key and account for everyone lost at each step;
2. recognise refused/don't-know codes, top-codes and other values that are not quantities;
3. distinguish a missing outcome from missing predictors, profile both, and argue which
   missingness mechanism (MCAR, MAR, MNAR) is most plausible, with evidence and limitations;
4. build a `ColumnTransformer` pipeline that imputes, encodes and scales inside cross-validation;
5. decide what to do with unusual values using knowledge of the measurement;
6. handle an imbalanced outcome and choose metrics that match it;
7. explain what a survey weight changes, and what an unweighted classroom analysis can and cannot claim.

**Prerequisites.** Weeks 3–5: train/test splits, cross-validation, scikit-learn
`Pipeline`, baselines, and the Week 4 metrics (sensitivity, specificity, PPV, ROC AUC).
Python with pandas at the level of the Week 2–5 labs.

**How the cells work.** Cells marked *provided* run as they are: read them, run
them, and make sure you understand them. Cells containing `# your code here` are
yours. Before some tasks you are asked to **predict** a result first: write the
prediction down before you run anything; it is the fastest way to find out what
you misunderstand.

**Structure.** **Tasks 1–9 are the core lab.** Some are short (Tasks 2 and 6).
A checkpoint after Task 9 marks the end of the core; three optional extensions
and a final reflection follow.

**Rules for the whole lab.** One split, made once, in the cell after Task 3. Every
choice is made with cross-validation on the training rows. The test rows are used
**once**, in Task 9. Anything that learns from data (imputers, scalers, encoders,
outlier detectors, resamplers) is fitted on training rows only.

**Practice, not assessment.** This lab is not part of Assignment 1 and uses a
different dataset. It practises skills Assignment 1 also needs, but the decisions
and justifications in your assignment must be made for *your* data, with *your*
numbers. Do not copy this lab's preprocessing into the assignment without
justifying it there.

**The Week 6 page is your interactive companion.** Its data-quality explorer
damages a real dataset in a known way and lets you measure the cost of each
repair; use it alongside Tasks 3–7.

**Deliverable.** This notebook, run top to bottom (**Kernel → Restart & Run All**),
with every question answered in the markdown cells using your own numbers.

### Setup

Run this cell first. It loads the nine files and nothing else.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (needed before the next import)
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import IsolationForest
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score, precision_score,
                             accuracy_score, balanced_accuracy_score, confusion_matrix)

pd.set_option("display.max_columns", 30)
SEED = 42
DATA = Path("data")
files = {name: pd.read_csv(DATA / f"nhanes_L_{name}.csv")
         for name in ["demo", "ghb", "bmx", "bpxo", "tchol", "dr1tot", "dr2tot", "smq", "diq"]}
demo, ghb = files["demo"], files["ghb"]
for name, d in files.items():
    print(f"{name:7s} {d.shape[0]:6,d} rows  {d.shape[1]:2d} columns: {', '.join(d.columns[1:])}")

**Variable names.** NHANES names are terse; `data/README.md` has the full
list. The ones you need most: `RIDAGEYR` age in years, `RIAGENDR` sex,
`RIDRETH3` race and Hispanic origin, `DMDEDUC2` education (adults), `INDFMPIR`
family income-to-poverty ratio, `RIDSTATR` interview only (1) or interviewed and
examined (2), `LBXGH` HbA1c (%), `BMXBMI` body-mass index, `BMXWAIST` waist (cm),
`BPXOSY1–3` three systolic readings (mmHg), `LBXTC` total cholesterol (mg/dL),
`DR1TKCAL` / `DR1TSUGR` energy (kcal) / total sugars (g) from the day-1 dietary
recall, `DR1DRSTZ` recall status, `SMQ020` smoked at least 100 cigarettes in life,
`DIQ010` ever told by a doctor you have diabetes, `WTPH2YR` the survey weight for
the blood-draw subsample.

## Task 1 — From nine files to one table

NHANES releases each component (interview, examination, laboratory, diet) as a
separate file. `SEQN` links them.

1. Check that `SEQN` is unique in every file.
2. Left-join `demo` with `ghb` on `SEQN`. Count: all participants; adults aged
   20 or over; adults who were examined (`RIDSTATR == 2`); adults with a
   measured HbA1c (`LBXGH` not missing).
3. Now inner-join `demo` with `ghb`, `bmx`, `bpxo`, `tchol`, `dr1tot` and `smq`,
   and count the adults in the result. Compare with your counts in step 2.

Step 1 is provided. **Predict first:** will the inner join in step 3 keep more,
fewer or the same number of adults as "adults with a measured HbA1c"?

In [ ]:
# provided: one row per participant in every file?
print({name: d.SEQN.is_unique for name, d in files.items()})

In [ ]:
# your code here


**Question 1.1.** Where are participants lost between "all participants" and
"adults with HbA1c"? For each step, say whether the loss is by design or by
circumstance.

**Question 1.2.** The inner join kept some adults whose HbA1c or BMI is missing.
How can a row survive an inner join and still have no value? What does this tell
you about "the row exists" versus "the value exists"?

**Question 1.3.** Why should the outcome file decide which rows enter the
modelling table, rather than an inner join of every file?

*Your answer here.*

## Task 2 — Read the codebook before you read the data (short)

1. Print `value_counts(dropna=False)` of `DMDEDUC2` and `SMQ020` for the adults
   with HbA1c. NHANES codes 7 as "refused" and 9 as "don't know". How many such
   codes are there?
2. Count adults with `RIDAGEYR == 80` and with `INDFMPIR == 5`. Both are
   **top-codes**: 80 means "80 or older", 5 means "5 or more".
3. Print the counts of `DR1DRSTZ` (1 = reliable and complete recall,
   2 = not reliable or not meeting minimum criteria, 5 = not done) and check
   whether any adult with status other than 1 has a value of `DR1TKCAL`.
4. Systolic blood pressure was measured three times. Make `SBP`, the mean of the
   available readings `BPXOSY1–3` for each participant.

In [ ]:
# your code here


**Question 2.1.** What would a model learn if `DMDEDUC2 = 9` ("don't know")
were left in as a number next to the education levels 1–5?

**Question 2.2.** Is a participant with `RIDAGEYR = 80` an outlier? Is the value
of 80 correct?

**Question 2.3.** `SBP` is computed from each participant's own three readings
before the train/test split. Why is this not leakage, when fitting a scaler before
the split is?

*Your answer here.*

### Checkpoint A (provided): the eligible adults and the analysis table

The cell below builds two tables with the decisions of Tasks 1 and 2.

* `eligible`: **every** adult aged 20+, with or without an HbA1c result, with two
  flags, `examined` and `has_hba1c`. It is made *before* the outcome filter so that
  Task 3 can ask who is missing the outcome. It is never used for modelling.
* `data`: the **modelling table**, the eligible adults with an HbA1c result, a left
  join of the predictor files, `SBP` as the mean of the available readings, and the
  codes 7 and 9 set to missing. The outcome is `diabetes_range = 1` if HbA1c ≥ 6.5%.
  Every later task uses `data`.

The predictors are grouped by type. Check that you agree with each grouping:
**numeric** (measured quantities), **ordinal** (ordered categories: education
1 = less than 9th grade … 5 = college graduate or above), **nominal** (categories
without an order: sex, race and Hispanic origin, ever smoked). Compare the shape
with your own table from Tasks 1–2.

In [ ]:
# 1. every eligible adult, before the outcome filter (Task 3 studies who has no HbA1c)
eligible = demo[demo.RIDAGEYR >= 20].merge(ghb, on="SEQN", how="left", validate="1:1")
eligible["examined"] = eligible.RIDSTATR == 2
eligible["has_hba1c"] = eligible.LBXGH.notna()

# 2. the modelling table: eligible adults WITH an HbA1c result, plus the predictors
data = eligible[eligible.has_hba1c].drop(columns=["examined", "has_hba1c"])
for name in ["bmx", "bpxo", "tchol", "dr1tot", "smq"]:
    data = data.merge(files[name], on="SEQN", how="left")
data["SBP"] = data[["BPXOSY1", "BPXOSY2", "BPXOSY3"]].mean(axis=1)
data["DMDEDUC2"] = data.DMDEDUC2.where(data.DMDEDUC2 <= 5)    # 7 refused, 9 don't know -> NaN
data["SMQ020"] = data.SMQ020.where(data.SMQ020 <= 2)
data["diabetes_range"] = (data.LBXGH >= 6.5).astype(int)
data = data.reset_index(drop=True)

NUMERIC = ["RIDAGEYR", "INDFMPIR", "BMXBMI", "BMXWAIST", "SBP", "LBXTC", "DR1TKCAL", "DR1TSUGR"]
ORDINAL = ["DMDEDUC2"]
NOMINAL = ["RIAGENDR", "RIDRETH3", "SMQ020"]
FEATURES = NUMERIC + ORDINAL + NOMINAL
TARGET = "diabetes_range"
print("eligible adults:", len(eligible), "| with HbA1c:", int(eligible.has_hba1c.sum()),
      "| without:", int((~eligible.has_hba1c).sum()))
print("modelling table:", data.shape, "| positives:", data[TARGET].sum(),
      f"| prevalence {data[TARGET].mean():.3f}")

## Task 3 — Profile the missingness, then argue about why

Two different things can be missing, and they are handled differently.

* **Outcome missingness.** An eligible adult has no HbA1c result. That person is not
  in `data` at all: we can neither train nor evaluate on them, and filling in an
  outcome would invent labels. It decides **whom the model describes**. You cannot
  see it in `data` (every row there has an outcome), so study it in `eligible`.
* **Predictor missingness.** An adult in `data` has an HbA1c result but, for
  example, no income. The row stays, and the pipeline handles the gap (Task 4).
  Study it in `data`.

Describing missingness learns no model parameter, so all rows are used here.

**Part A — the outcome (use `eligible`).**

1. *(provided)* Count the eligible adults with and without an HbA1c result, and
   split those without into "not examined" and "examined, no HbA1c result".
2. Merge `files["diq"]` into `eligible` and compare the fraction who report a
   doctor's diagnosis of diabetes (`DIQ010 == 1`; treat 2 and 3 as no, 7 and 9 as
   missing) and the mean age: first between adults with and without HbA1c, then
   for each of the three groups from step 1.

**Part B — the predictors (use `data`).**

3. *(provided)* The fraction missing for each feature, and the number of rows with
   no missing feature.
4. **Income.** Show the fraction of `INDFMPIR` missing at each education level,
   and the prevalence of the outcome among participants with and without income.
5. **Diet.** Compare the mean age and the outcome prevalence of participants with
   and without a day-1 recall (`DR1TKCAL`).

**Predict first:** will adults *without* an HbA1c result report a diabetes diagnosis
more often or less often than adults with one?

In [ ]:
# provided: Part A, step 1. Outcome missingness, counted on every eligible adult.
eligible["stage"] = np.select([eligible.has_hba1c, eligible.examined],
                              ["has HbA1c", "examined, no HbA1c result"], "not examined")
print(eligible.stage.value_counts().to_string())
print(f"no HbA1c: {(~eligible.has_hba1c).sum()} of {len(eligible)} eligible adults "
      f"({(~eligible.has_hba1c).mean():.1%}); none of them is in `data`")

In [ ]:
# your code here


In [ ]:
# provided: Part B, step 3. Predictor missingness, within the modelling table.
miss = data[FEATURES].isna().mean().sort_values(ascending=False)
print(miss.round(3).to_string())
print("rows with no missing feature:", int(data[FEATURES].notna().all(axis=1).sum()), "of", len(data))

In [ ]:
# your code here


**Question 3.1 (outcome).** Is the modelling table `data` a random subset of the
eligible adults? Using your Part A numbers, including the two groups without HbA1c
separately, argue which mechanism is most plausible for the missing outcome: missing
completely at random (MCAR), at random given observed variables (MAR), or not at random
(MNAR). State your evidence and what your tables cannot tell you.

**Question 3.2 (predictors).** Make the same argument for **income** and **diet**,
citing your Part B numbers.

**Question 3.3.** Which kind of missingness can the imputation in Task 4 address, and
which can it not? What can an analyst do about the other kind?

**Question 3.4.** Can any table you made *prove* that a variable is MAR rather
than MNAR? What would you need to know?

**Question 3.5.** What would a model trained only on rows with no missing feature
learn about, and whom would it fail to describe?

*Your answer here.*

### The split (provided): made once, used for the rest of the lab

A stratified 75/25 split with `SEED = 42`, and the 5-fold cross-validation used for
every choice. After this cell, `X_test` and `y_test` are not touched until Task 9.

In [ ]:
X, y = data[FEATURES], data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y,
                                                    random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"train {len(X_train):,} rows ({y_train.sum()} positive, {y_train.mean():.3f}) | "
      f"test {len(X_test):,} rows ({y_test.sum()} positive, {y_test.mean():.3f})")

## Task 4 — Three ways to handle missing values, inside a Pipeline

**(a)** Write a function `nhanes_preprocessor(numeric_imputer)` that returns a
`ColumnTransformer` with two parts:

* numeric and ordinal columns: `numeric_imputer`, then `StandardScaler()`;
* nominal columns: `SimpleImputer(strategy="most_frequent")`, then
  `OneHotEncoder(handle_unknown="ignore", drop="if_binary")`.

**(b)** A helper `cv_scores(pipe, X, y)` is provided below. Read it: it runs
`cross_validate` with the `cv` object above and returns the mean and SD of `roc_auc`,
`average_precision`, `recall` and `balanced_accuracy`.

**(c)** With `LogisticRegression(max_iter=1000)` as the model, compare by
cross-validation on the training rows:

1. **complete case**: median imputer, but only the training rows with no missing
   feature;
2. **median**: `SimpleImputer(strategy="median")`, all training rows;
3. **iterative + indicator**: `IterativeImputer(random_state=SEED, add_indicator=True)`,
   all training rows.

Add a majority-class baseline (`DummyClassifier(strategy="most_frequent")`). Show one
table with the number of rows and the prevalence used by each. For the complete-case
variant, also report the fraction of training and of test rows it can score.

In [ ]:
# provided: the evaluation helper (step b) and the baseline every model is compared with
SCORING = ["roc_auc", "average_precision", "recall", "balanced_accuracy"]

def cv_scores(pipe, X, y):
    """5-fold CV on the rows given; every step of `pipe` is refitted on each training fold."""
    res = cross_validate(pipe, X, y, cv=cv, scoring=SCORING)
    out = {"n": len(X), "prevalence": y.mean()}
    for s in SCORING:
        out[s] = res["test_" + s].mean()
        out[s + "_sd"] = res["test_" + s].std()
    return out

results = {"baseline (majority)": cv_scores(DummyClassifier(strategy="most_frequent"), X_train, y_train)}
pd.Series(results["baseline (majority)"]).round(3)

**(a) is yours:** write `nhanes_preprocessor` in the next cell.

In [ ]:
# your code here


In [ ]:
# provided: your preprocessor + logistic regression, used from here on
def logreg(imputer, **kw):
    return make_pipeline(nhanes_preprocessor(imputer), LogisticRegression(max_iter=1000, **kw))

**(c) is yours.** **Predict first:** which variant will have the highest CV ROC
AUC, and will the differences be larger or smaller than the fold-to-fold SD?

In [ ]:
# your code here


**Question 4.1.** Which handling gives the best mean ROC AUC and average
precision? Is the difference larger than the cross-validation SD? Would you claim
a winner?

**Question 4.2.** The complete-case scores look similar to the others. Give two
reasons why they are still not comparable with them, and one reason the
complete-case model is unusable in practice.

**Question 4.3.** Which handling would you defend in a paper, and why? Your answer
should mention the population, not only the score.

**Question 4.4.** Why does the imputer have to be inside the Pipeline for
`cross_validate` to give an honest estimate?

*Your answer here.*

## Task 5 — Outliers: look before you delete

Work on the **training rows only**. Never remove rows from the test set: that
would delete the hard cases and then report how well you did on the easy ones.

1. For `BMXBMI`, `DR1TKCAL` and `SBP`, compute the IQR fences
   (Q1 − 1.5 IQR, Q3 + 1.5 IQR). For each, report the fences, the number of
   training rows outside them, and the outcome prevalence inside and outside.
2. Fit `IsolationForest(contamination=0.02, random_state=SEED)` on the numeric
   training features after median imputation and standardisation. Report the number
   flagged and the prevalence among flagged and unflagged rows.
3. Show three flagged training participants: the highest `DR1TKCAL`, the highest
   `BMXBMI` and the lowest `DR1TKCAL`, with all their features.

**Predict first:** among participants outside the BMI fences, will diabetes-range
HbA1c be more common, less common or about as common as among the rest?

In [ ]:
# provided: Tukey's fences, learned from the values you pass in (pass training rows only)
def iqr_fences(values):
    q1, q3 = values.quantile([0.25, 0.75])
    return q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)

print("BMI fences from training rows:", [round(v, 2) for v in iqr_fences(X_train.BMXBMI)])

In [ ]:
# your code here


**Question 5.1.** What would deleting the BMI "outliers" do to the training
data? Use the prevalence numbers.

**Question 5.2.** For each of your three participants, decide **keep**, **cap** or
**set to missing**, in one sentence each, saying whether the value is more likely a
measurement or reporting error, a real but rare value, or the clinically
interesting case.

**Question 5.3.** The IQR rule flags nothing below zero kcal and a lower fence below
zero. What does that tell you about using the IQR rule on a skewed variable?

*Your answer here.*

## Task 6 — Encoding and scaling: two controlled checks (short)

Change **one thing at a time** against the median pipeline from Task 4.

1. **Integer-coded nominal variables.** Replace the preprocessor by a single
   median imputer + `StandardScaler` applied to *all* features, so that `RIDRETH3`
   (codes 1, 2, 3, 4, 6, 7) is treated as a number. Compare cross-validated ROC AUC
   and average precision with the one-hot version.
2. **No scaling.** Build the Task 4 median pipeline without the `StandardScaler`,
   with the default `LogisticRegression()` (100 iterations). Count the
   `ConvergenceWarning`s raised during cross-validation and compare the scores. Print
   the training range (min, max) of `RIDAGEYR`, `SBP` and `DR1TKCAL`.

In [ ]:
# your code here


**Question 6.1.** The integer-coded version scores only slightly lower. Is it
therefore acceptable? What does its coefficient for `RIDRETH3` assume?

**Question 6.2.** Why did the unscaled logistic regression fail to converge, and why
does scaling matter to a *regularised* model even when it converges? Name one model
from this module for which scaling would make no difference, and say why.

*Your answer here.*

## Task 7 — Class imbalance: accuracy, class weights and the threshold

Only about one in eight participants is positive.

1. What accuracy does the majority-class baseline achieve on the training rows? Use
   your Task 4 table.
2. Compare by cross-validation the Task 4 median pipeline with the same pipeline using
   `LogisticRegression(max_iter=1000, class_weight="balanced")`. Report ROC AUC, average
   precision, recall and balanced accuracy (both at the default threshold 0.5).

In [ ]:
# your code here


**Question 7.1.** Why is accuracy a poor headline metric for this question?

**Question 7.2.** Class weighting changed recall enormously but ROC AUC hardly at all.
Explain why, using what ROC AUC measures. How is `class_weight="balanced"` related to
moving the decision threshold (Week 4)?

**Question 7.3.** Resampling (duplicating minority rows, or SMOTE) is an alternative to
class weights. Where in the workflow must it happen, and why? (Extension E1 tests this.)

*Your answer here.*

## Task 8 — Survey weights: whom does the sample describe?

NHANES is not a simple random sample: some groups are deliberately oversampled and
response differs between groups. Each participant's weight is (roughly) the number of
people in the US civilian non-institutionalised population they represent. HbA1c was
measured in the blood-draw subsample, so the right weight is `WTPH2YR`.

1. Compute the unweighted prevalence of `diabetes_range` in `data`, and the weighted
   prevalence `np.average(data[TARGET], weights=data.WTPH2YR)`.
2. For the age groups 20–39, 40–59 and 60–80, show the share of the sample, the
   weighted share (sum of weights in the group / total weight) and the prevalence.

**Predict first:** older adults were over-represented in this cycle's sample. Will
the weighted prevalence be higher or lower than the unweighted one? Then run the
provided step 1.

In [ ]:
# provided: step 1. A weighted mean is a population estimate; it is description, not modelling.
print(f"unweighted prevalence {data[TARGET].mean():.4f} | weighted (WTPH2YR) "
      f"{np.average(data[TARGET], weights=data.WTPH2YR):.4f}")

In [ ]:
# your code here


**Question 8.1.** In one paragraph: why is the weighted prevalence lower than
the unweighted one? Use your age table.

**Question 8.2.** Your model was trained and evaluated without weights. What population
do its cross-validation scores describe? Name one thing (beyond a point estimate) that a
proper survey analysis would also need.

*Your answer here.*

## Task 9 — Choose by cross-validation, test once, write the record

**The decision rule, fixed before looking:** the purpose is to flag adults for a
confirmatory test, so a missed case costs more than a false alarm. Among the pipelines
that can score every participant (Task 4 median, Task 4 iterative + indicator, Task 7
balanced), take the best mean CV **balanced accuracy** at threshold 0.5, and keep the
simplest pipeline within one SD of it (simplest first: median, median + balanced weights,
iterative + indicator).

1. Apply the rule to your `results` and name the chosen pipeline.
2. Fit it on **all training rows**, and evaluate it **once** on the test rows with the
   provided `test_report`: ROC AUC, average precision, accuracy, recall, specificity,
   precision (PPV) and the confusion matrix. Put the test numbers next to its CV numbers.

In [ ]:
# provided: everything reported about the test set, computed in one place, once
def test_report(y_true, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    scores = {"roc_auc": roc_auc_score(y_true, p), "average_precision": average_precision_score(y_true, p),
              "accuracy": accuracy_score(y_true, pred), "balanced_accuracy": balanced_accuracy_score(y_true, pred),
              "recall": recall_score(y_true, pred), "specificity": tn / (tn + fp),
              "precision": precision_score(y_true, pred, zero_division=0)}
    return scores, (tp, fn, fp, tn)

In [ ]:
# your code here


**Question 9.1.** How do the test results compare with the CV results? Should
the test result change your choice of pipeline?

**Question 9.2.** Of the participants the model flags, what fraction actually have
HbA1c ≥ 6.5%? Explain this number with the prevalence (Week 4), and say what a flag
should and should not mean for a participant.

### The reproducibility record

Complete the table. Someone else should be able to repeat your Task 9 result from it.

| Item | Your entry |
|---|---|
| Data (files, cycle, rows after each step of Task 1) | |
| Population and outcome definition | |
| Codes and cleaning (7/9, top-codes, SBP) | |
| Split (method, seed, sizes, prevalence) | |
| Pipeline (every step and its settings) | |
| What was fitted on what | |
| Decision rule and the CV evidence for the choice | |
| Test results (once) | |
| What the result does **not** show (population, weights, diagnosis) | |

In [ ]:
import sklearn, sys
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__)

*Your answer here.*

---
### Checkpoint: end of the core lab

You have taken raw survey files to a documented, evaluated pipeline, and every
decision has a written reason. The extensions are optional.

## Extension E1 — Resampling before the split

A tempting "fix" for imbalance: duplicate positive rows at random until the classes are
equal, then cross-validate. Test it.

1. Write `oversample(X, y, seed)` that returns X and y with randomly chosen positive rows
   duplicated (with `np.random.default_rng(seed)`) until the classes are equal.
2. **Wrong:** oversample all training rows, then run the 5-fold `cv` on the result.
3. **Right:** for each fold of `cv` on the original training rows, oversample only the
   training part, fit, and score the untouched validation part.
4. Do both for an unpruned `DecisionTreeClassifier(random_state=SEED)` and for the Task 4
   median logistic regression, using the median preprocessor. Report ROC AUC, average
   precision, recall, precision and the validation prevalence.

In [ ]:
# your code here


**Question E1.1.** Explain the tree's two results. What fraction of each validation
fold in the "wrong" version could the tree have memorised?

**Question E1.2.** Logistic regression's ROC AUC hardly changed, but its precision did.
Why? Is the "wrong" version acceptable for logistic regression?

*Your answer here.*

## Extension E2 — One day of diet is not a habit

Every participant was eligible for two 24-hour dietary recalls. In this cycle both were
collected by telephone, the second 3 to 10 days after the first. For participants
in `data` with both, plot day-1 against day-2 energy intake (`files["dr2tot"].DR2TKCAL`),
and report the correlation and the median absolute difference between the two days.

In [ ]:
# your code here


**Question E2.1.** What does the correlation say about using one recall as a measure
of a person's *usual* intake? What happens to an estimated association between usual
intake and HbA1c when the exposure is measured this noisily? Is this something a better
model could fix?

*Your answer here.*

## Extension E3 (challenge, optional software) — SMOTE inside the pipeline

SMOTE creates synthetic minority rows by interpolating between neighbouring minority rows.
It is in the `imbalanced-learn` package, which is **not** part of the course environment.
If you have it (`pip install imbalanced-learn`), build
`imblearn.pipeline.make_pipeline(nhanes_preprocessor(SimpleImputer(strategy="median")),
SMOTE(random_state=SEED), LogisticRegression(max_iter=1000))` and cross-validate it with
`cv_scores`. Why does this need `imblearn`'s Pipeline rather than scikit-learn's? Is a
synthetic participant interpolated between two real ones always biologically plausible?
(Think about a one-hot encoded column.)

In [ ]:
# your code here


*Your answer here.*

## Final reflection

Answer in a short paragraph each.

**R1.** List the preparation decisions you made today that changed *who the model is
about*, as opposed to *how well it scores*. Which of them would a reader of a paper be
least likely to notice?

**R2.** A colleague proposes replacing your logistic regression with a large pretrained
model "because it handles messy data". Which of today's problems would it fix, and which
would it not? Consider the missing outcomes, the dietary recall, the weights and the
top-codes.

*Your answer here.*